In [1]:
import sys
from pathlib import Path

# make the repo root importable so `src` works from any folder
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'paths.py').exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
import yaml
from src.paths import EXTRACTED, INTERIM, REPORTS, CONFIG

# one series = one (time, metric, gpu, job); every step below uses this key
KEY = ["timeUtc", "metric", "gpu", "job"]


In [2]:
planes = pd.read_csv(REPORTS / "metric_planes.csv").set_index("metric")
kept_metrics = planes.index[planes["keep"]]

print("kept metrics:", len(kept_metrics))      # expect 247
planes["kind"].value_counts()


kept metrics: 247


kind
gauge      190
counter     70
Name: count, dtype: int64

In [3]:
stem = "ggpu142_2025-02-17_gpus-fallen-off-bus"

raw = pd.read_csv(
    EXTRACTED / f"{stem}_tidy.csv",
    usecols=["timeUtc", "node", "metric", "value", "gpu", "job"],
    dtype=str,                                   # labels as text, no mixed-type guessing
)
raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
raw[["gpu", "job"]] = raw[["gpu", "job"]].fillna("")   # NaN keys would be dropped by groupby

rows_before = len(raw)
raw = raw[raw["metric"].isin(kept_metrics)]

print("raw rows:", rows_before, "-> after keep filter:", len(raw))
print("values that are not numbers:", raw["value"].isna().sum())
raw.head()


raw rows: 877742 -> after keep filter: 818848
values that are not numbers: 0


,timeUtc,node,metric,value,gpu,job
0,2025-02-10 00:00:00,node_6a5ebc855306,DCGM_FI_DEV_CORRECTABLE_REMAPPED_ROWS,0.0,2,prometheus.scrape.default
1,2025-02-10 00:10:00,node_6a5ebc855306,DCGM_FI_DEV_CORRECTABLE_REMAPPED_ROWS,0.0,2,prometheus.scrape.default
2,2025-02-10 00:20:00,node_6a5ebc855306,DCGM_FI_DEV_CORRECTABLE_REMAPPED_ROWS,0.0,2,prometheus.scrape.default
3,2025-02-10 00:30:00,node_6a5ebc855306,DCGM_FI_DEV_CORRECTABLE_REMAPPED_ROWS,0.0,2,prometheus.scrape.default
4,2025-02-10 00:40:00,node_6a5ebc855306,DCGM_FI_DEV_CORRECTABLE_REMAPPED_ROWS,0.0,2,prometheus.scrape.default


In [4]:
# counters are summed, gauges averaged (decided in metric_planes.csv)
raw["kind"] = raw["metric"].map(planes["kind"])

counters = raw[raw["kind"] == "counter"].groupby(KEY + ["node"])["value"]
gauges = raw[raw["kind"] == "gauge"].groupby(KEY + ["node"])["value"]

combined = pd.concat([
    # min_count=1: if all rows are NaN the result stays NaN instead of becoming 0
    pd.DataFrame({"value": counters.sum(min_count=1), "n_rows": counters.size()}),
    pd.DataFrame({"value": gauges.mean(), "n_rows": gauges.size()}),
]).reset_index()

combined["timeUtc"] = pd.to_datetime(combined["timeUtc"], format="%Y-%m-%d %H:%M:%S")
combined["source_file"] = stem

print("rows:", len(raw), "->", len(combined))
combined.head()


rows: 818848 -> 358996


,timeUtc,metric,gpu,job,node,value,n_rows,source_file
0,2025-02-10,DCGM_FI_DEV_PCIE_REPLAY_COUNTER,0,prometheus.scrape.default,node_6a5ebc855306,0.000000e+00,1,ggpu142_2025-02-17_gpus-fallen-off-bus
1,2025-02-10,DCGM_FI_DEV_PCIE_REPLAY_COUNTER,1,prometheus.scrape.default,node_6a5ebc855306,0.000000e+00,1,ggpu142_2025-02-17_gpus-fallen-off-bus
2,2025-02-10,DCGM_FI_DEV_PCIE_REPLAY_COUNTER,2,prometheus.scrape.default,node_6a5ebc855306,0.000000e+00,1,ggpu142_2025-02-17_gpus-fallen-off-bus
3,2025-02-10,DCGM_FI_DEV_PCIE_REPLAY_COUNTER,3,prometheus.scrape.default,node_6a5ebc855306,0.000000e+00,1,ggpu142_2025-02-17_gpus-fallen-off-bus
4,2025-02-10,DCGM_FI_DEV_TOTAL_ENERGY_CONSUMPTION,0,prometheus.scrape.default,node_6a5ebc855306,1.894524e+11,1,ggpu142_2025-02-17_gpus-fallen-off-bus


In [5]:
# 1. number of series: list each (metric, gpu, job) once.
#    (drop_duplicates here only lists names; it never touches the data)
series = combined[["metric", "gpu", "job"]].drop_duplicates()
print("series:", len(series))                                       # expect about 316

# 2. exactly one value per key
print("duplicate keys:", combined.duplicated(KEY).sum())            # expect 0

# 3. GPUs and jobs present
print("GPUs:", sorted(combined.loc[combined["gpu"] != "", "gpu"].unique()))   # expect 4 GPUs
print("jobs:", sorted(combined["job"].unique()))                    # expect 4 jobs

# 4. how many metrics were actually combined (n_rows > 1 somewhere)
print("metrics with combined rows:", combined.loc[combined["n_rows"] > 1, "metric"].nunique())


series: 312
duplicate keys: 0
GPUs: ['0', '1', '2', '3']
jobs: ['', 'integrations/unix', 'ipmi_remote', 'prometheus.scrape.default']
metrics with combined rows: 86


In [6]:
m = "node_network_receive_bytes_total"
t = raw.loc[raw["metric"] == m, "timeUtc"].iloc[0]

print("BEFORE: raw rows at", t)
print(raw[(raw["metric"] == m) & (raw["timeUtc"] == t)][["metric", "job", "value"]])

print("\nAFTER: combined")
print(combined[(combined["metric"] == m) & (combined["timeUtc"] == pd.Timestamp(t))][["metric", "job", "value", "n_rows"]])


BEFORE: raw rows at 2025-02-10 00:00:00
                                  metric                job         value
553840  node_network_receive_bytes_total  integrations/unix  0.000000e+00
555089  node_network_receive_bytes_total  integrations/unix  2.770644e+09
556338  node_network_receive_bytes_total  integrations/unix  7.003511e+08
557587  node_network_receive_bytes_total  integrations/unix  1.687564e+10
558836  node_network_receive_bytes_total  integrations/unix  1.140346e+09

AFTER: combined
                              metric                job         value  n_rows
53  node_network_receive_bytes_total  integrations/unix  2.148698e+10       5


In [7]:
def load_and_combine(stem):
    """Load one incident file and return one value per (time, metric, gpu, job)."""
    raw = pd.read_csv(
        EXTRACTED / f"{stem}_tidy.csv",
        usecols=["timeUtc", "node", "metric", "value", "gpu", "job"],
        dtype=str,
    )
    raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
    raw[["gpu", "job"]] = raw[["gpu", "job"]].fillna("")
    raw = raw[raw["metric"].isin(kept_metrics)]
    raw["kind"] = raw["metric"].map(planes["kind"])

    counters = raw[raw["kind"] == "counter"].groupby(KEY + ["node"])["value"]
    gauges = raw[raw["kind"] == "gauge"].groupby(KEY + ["node"])["value"]
    combined = pd.concat([
        pd.DataFrame({"value": counters.sum(min_count=1), "n_rows": counters.size()}),
        pd.DataFrame({"value": gauges.mean(), "n_rows": gauges.size()}),
    ]).reset_index()

    combined["timeUtc"] = pd.to_datetime(combined["timeUtc"], format="%Y-%m-%d %H:%M:%S")
    combined["source_file"] = stem
    return combined

# the function must give exactly the same result as the cells above
check = load_and_combine(stem)
print("same as step-by-step result:", len(check) == len(combined) and check["value"].sum() == combined["value"].sum())


same as step-by-step result: True


In [8]:
import numpy as np

# files to leave out, decided earlier (see config/preprocess.yaml)
config = yaml.safe_load((CONFIG / "preprocess.yaml").read_text(encoding="utf-8"))
excluded = set(config["excluded_files"])

files = pd.DataFrame({"stem": [p.name.replace("_tidy.csv", "") for p in EXTRACTED.glob("ggpu*_tidy.csv")]})
files = files[~files["stem"].isin(excluded)]

# file name = <host>_<incident date>_<category>
files["host"] = files["stem"].str.split("_").str[0]
files["fileDate"] = pd.to_datetime(files["stem"].str.split("_").str[1])
files = files.sort_values(["host", "fileDate"]).reset_index(drop=True)

print("files:", len(files), "| nodes:", files["host"].nunique())    # expect 15 files, 7 nodes
files.groupby("host")["stem"].count()
# expect: ggpu121 1, ggpu129 3, ggpu139 1, ggpu142 5, ggpu143 1, ggpu144 1, ggpu149 3
import numpy as np

# files to leave out, decided earlier (see config/preprocess.yaml)
config = yaml.safe_load((CONFIG / "preprocess.yaml").read_text(encoding="utf-8"))
excluded = set(config["excluded_files"])

files = pd.DataFrame({"stem": [p.name.replace("_tidy.csv", "") for p in EXTRACTED.glob("ggpu*_tidy.csv")]})
files = files[~files["stem"].isin(excluded)]

# file name = <host>_<incident date>_<category>
files["host"] = files["stem"].str.split("_").str[0]
files["fileDate"] = pd.to_datetime(files["stem"].str.split("_").str[1])
files = files.sort_values(["host", "fileDate"]).reset_index(drop=True)

print("files:", len(files), "| nodes:", files["host"].nunique())    # expect 15 files, 7 nodes
files.groupby("host")["stem"].count()
# expect: ggpu121 1, ggpu129 3, ggpu139 1, ggpu142 5, ggpu143 1, ggpu144 1, ggpu149 3


files: 15 | nodes: 7
files: 15 | nodes: 7


host
ggpu121    1
ggpu129    3
ggpu139    1
ggpu142    5
ggpu143    1
ggpu144    1
ggpu149    3
Name: stem, dtype: int64

In [9]:
def compare_overlap(earlier, later, start, end):
    """Compare two files on the moments both cover. They are copies of the same
    monitoring data, so the values should agree (about 100%)."""
    a = earlier[earlier["timeUtc"].between(start, end)]
    b = later[later["timeUtc"].between(start, end)]
    both = a.merge(b, on=KEY, suffixes=("_earlier", "_later"))

    same_value = np.isclose(both["value_earlier"], both["value_later"], equal_nan=True)
    same_rows = both["n_rows_earlier"] == both["n_rows_later"]

    return {
        "earlier_file": ", ".join(a["source_file"].unique()),
        "later_file": b["source_file"].iloc[0],
        "overlap_start": start,
        "overlap_end": end,
        "points_earlier": len(a),
        "points_later": len(b),
        "points_in_both": len(both),
        "value_agreement_pct": round(100 * same_value.mean(), 3),
        "n_rows_agreement_pct": round(100 * same_rows.mean(), 3),
    }
def compare_overlap(earlier, later, start, end):
    """Compare two files on the moments both cover. They are copies of the same
    monitoring data, so the values should agree (about 100%)."""
    a = earlier[earlier["timeUtc"].between(start, end)]
    b = later[later["timeUtc"].between(start, end)]
    both = a.merge(b, on=KEY, suffixes=("_earlier", "_later"))

    same_value = np.isclose(both["value_earlier"], both["value_later"], equal_nan=True)
    same_rows = both["n_rows_earlier"] == both["n_rows_later"]

    return {
        "earlier_file": ", ".join(a["source_file"].unique()),
        "later_file": b["source_file"].iloc[0],
        "overlap_start": start,
        "overlap_end": end,
        "points_earlier": len(a),
        "points_later": len(b),
        "points_in_both": len(both),
        "value_agreement_pct": round(100 * same_value.mean(), 3),
        "n_rows_agreement_pct": round(100 * same_rows.mean(), 3),
    }


In [10]:
def merge_node(stems):
    """Join a node's files into one timeline. Files are in date order; each moment
    is taken from the earliest file that covers it, later files only add moments
    after the end of what is already covered."""
    parts, checks = [], []
    covered_until = None

    for stem in stems:
        data = load_and_combine(stem)
        start, end = data["timeUtc"].min(), data["timeUtc"].max()

        if covered_until is not None and start <= covered_until:
            # overlap: compare first, then keep only the new moments
            checks.append(compare_overlap(pd.concat(parts), data, start, covered_until))
            data = data[data["timeUtc"] > covered_until]

        parts.append(data)
        covered_until = end if covered_until is None else max(covered_until, end)
        print(f"  {stem:<52} kept {len(data):>9,} rows")

    merged = pd.concat(parts, ignore_index=True).sort_values(KEY).reset_index(drop=True)
    return merged, checks


In [11]:
test_stems = files.loc[files["host"] == "ggpu142", "stem"].tolist()
test, test_checks = merge_node(test_stems)

print("\nduplicate keys:", test.duplicated(KEY).sum())                 # expect 0
print("source files:", test["source_file"].nunique())                   # expect 5

# separate time blocks: a gap of more than a day starts a new block
ts = pd.Series(test["timeUtc"].unique()).sort_values()
print("time blocks:", (ts.diff() > pd.Timedelta("1D")).sum() + 1)       # expect 4 (Feb, Mar, Jul, Sep)

pd.DataFrame(test_checks)       # expect 1 row (03-19 vs 03-21), agreement about 100%


  ggpu142_2025-02-17_gpus-fallen-off-bus               kept   358,996 rows
  ggpu142_2025-03-19_gpu-problem                       kept   421,956 rows
  ggpu142_2025-03-21_gpus-fell-off-bus                 kept    89,856 rows
  ggpu142_2025-07-03_gpu-problem                       kept   443,916 rows
  ggpu142_2025-09-12_gpus-dropped-off-bus              kept   434,411 rows

duplicate keys: 0
source files: 5
time blocks: 4


,earlier_file,later_file,overlap_start,overlap_end,points_earlier,points_later,points_in_both,value_agreement_pct,n_rows_agreement_pct
0,ggpu142_2025-03-19_gpu-problem,ggpu142_2025-03-21_gpus-fell-off-bus,2025-03-14,2025-03-21 23:50:00,332100,332100,332100,100.0,100.0


In [12]:
INTERIM.mkdir(parents=True, exist_ok=True)
summary, all_checks = [], []

for host, group in files.groupby("host"):
    print(host)
    merged, checks = merge_node(group["stem"].tolist())
    assert merged.duplicated(KEY).sum() == 0, f"{host}: duplicate keys after merge"

    merged.to_parquet(INTERIM / f"{host}.parquet", index=False)

    ts = pd.Series(merged["timeUtc"].unique()).sort_values()
    summary.append({
        "host": host,
        "node": merged["node"].iloc[0],
        "n_files": len(group),
        "start": ts.min(),
        "end": ts.max(),
        "n_timestamps": len(ts),
        "time_blocks": (ts.diff() > pd.Timedelta("1D")).sum() + 1,
        "n_series": len(merged[["metric", "gpu", "job"]].drop_duplicates()),
        "n_gpus": merged.loc[merged["gpu"] != "", "gpu"].nunique(),
        "rows": len(merged),
    })
    all_checks += [{"host": host, **c} for c in checks]
    print()


ggpu121
  ggpu121_2025-02-10_gpu-error                         kept   251,529 rows

ggpu129
  ggpu129_2026-01-09_gpu-lost                          kept   353,138 rows
  ggpu129_2026-01-13_gpu-lost                          kept   180,126 rows
  ggpu129_2026-01-19_gpu-lost                          kept   205,324 rows

ggpu139
  ggpu139_2025-03-21_gpus-dropped-off-bus-maybe        kept   441,964 rows

ggpu142
  ggpu142_2025-02-17_gpus-fallen-off-bus               kept   358,996 rows
  ggpu142_2025-03-19_gpu-problem                       kept   421,956 rows
  ggpu142_2025-03-21_gpus-fell-off-bus                 kept    89,856 rows
  ggpu142_2025-07-03_gpu-problem                       kept   443,916 rows
  ggpu142_2025-09-12_gpus-dropped-off-bus              kept   434,411 rows

ggpu143
  ggpu143_2025-03-21_gpu-problem                       kept   442,516 rows

ggpu144
  ggpu144_2025-03-21_gpu-problem                       kept   441,680 rows

ggpu149
  ggpu149_2025-03-21_gpus-dropped-off-

In [13]:
summary = pd.DataFrame(summary)
overlap = pd.DataFrame(all_checks)

summary.to_csv(REPORTS / "merge_summary.csv", index=False)
overlap.to_csv(REPORTS / "merge_overlap_check.csv", index=False)

print("node files saved:", len(list(INTERIM.glob("*.parquet"))))        # expect 7
display(summary)
# expect: n_gpus 4 everywhere; n_series about 312-316;
#         time_blocks: ggpu142 4, ggpu149 3, all others 1
display(overlap)
# expect: overlaps only for ggpu129 (2 rows) and ggpu142 (1 row), agreement about 100%


node files saved: 7


,host,node,n_files,start,end,n_timestamps,time_blocks,n_series,n_gpus,rows
0,ggpu121,node_2b08d249c0ba,1,2025-02-03,2025-02-12 23:50:00,1440,1,312,4,251529
1,ggpu129,node_b1b91a387252,3,2026-01-02,2026-01-21 23:50:00,2690,2,316,4,738588
2,ggpu139,node_df3724dfcace,1,2025-03-14,2025-03-23 23:50:00,1440,1,312,4,441964
3,ggpu142,node_6a5ebc855306,5,2025-02-10,2025-09-14 23:50:00,6048,4,316,4,1749135
4,ggpu143,node_62a6cb9d8856,1,2025-03-14,2025-03-23 23:50:00,1440,1,312,4,442516
5,ggpu144,node_d4119f476626,1,2025-03-14,2025-03-23 23:50:00,1440,1,312,4,441680
6,ggpu149,node_039ab2df8a6a,3,2025-03-14,2026-01-21 23:50:00,4046,4,316,4,1145792


,host,earlier_file,later_file,overlap_start,overlap_end,points_earlier,points_later,points_in_both,value_agreement_pct,n_rows_agreement_pct
0,ggpu129,ggpu129_2026-01-09_gpu-lost,ggpu129_2026-01-13_gpu-lost,2026-01-06,2026-01-11 23:50:00,171122,171122,171122,100.0,100.0
1,ggpu129,ggpu129_2026-01-13_gpu-lost,ggpu129_2026-01-19_gpu-lost,2026-01-12,2026-01-15 23:50:00,180126,180126,180126,100.0,100.0
2,ggpu142,ggpu142_2025-03-19_gpu-problem,ggpu142_2025-03-21_gpus-fell-off-bus,2025-03-14,2025-03-21 23:50:00,332100,332100,332100,100.0,100.0
